# 02A — Processing AR cubes into corrected, analysis-ready cubes

This notebook is the **processing** step: it takes the raw cubes `01A_download_data.ipynb`
produced and writes corrected ones. It draws nothing — all plotting and analysis lives in
`03A_data_analisis.ipynb`, which reads what this notebook writes.

For every region (`data/raw/NOAA_<noaa>_<date>/region_01_*_cube.fits`) it:

1. Divides the continuum by its **limb-darkening** factor, the fifth Castellanos Durán &
   Kleint (2020) correction (`src/limb_darkening.py`).
2. Calibrates the dopplergram with the four Castellanos Durán et al. (2021) zero-point
   corrections, giving absolute m/s (`src/doppler_calibration.py`).
3. Places the continuum / magnetogram / dopplergram cubes on one **uniform time grid**,
   NaN-filling any slot a series has no frame for.
4. Segments each continuum frame into **umbra**, **penumbra** and **Qsun**, thresholding
   against that frame's own quiet-sun intensity.
5. Removes the quiet-sun background from the magnetogram by subtracting a fitted plane.
6. Builds the **hot spot** — the strong-field core inside the sunspot footprint.
7. Writes everything to `data/processed/NOAA_<noaa>_<date>/`.

Steps 1 and 2 run *before* step 3 and are applied per frame, because both depend on where
the tracked box is pointing at that instant.

## Outputs

| File | Contents |
| --- | --- |
| `region_01_continuum_cube.fits` | limb-darkening corrected, still DN/s |
| `region_01_magnetogram_corrected_cube.fits` | quiet-sun plane removed |
| `region_01_dopplergram_calibrated_cube.fits` | absolute LOS velocity, m/s |
| `region_01_masks_cube.fits` | `uint8` bitmask — bit0 umbra, bit1 penumbra, bit2 hot spot |
| `region_01_qsun_means.fits` | per-frame quiet-sun mean B and v, `I_qs`, `C_MEAN`, `PRESENT_*` |

All carry the `TIMESTAMPS` extension and `HISTORY` cards recording which corrections were
applied, and all open directly in **DS9** as cubes with a frame slider.

## The time axis

The three series are joined on a grid that is **uniform by construction**, covering the
union of their timestamps, rather than on the timestamps they happen to share. A frame a
series is missing becomes a **NaN frame in the right slot**, and the `PRESENT_CONT` /
`PRESENT_MAG` / `PRESENT_DOP` columns of the qsun table say which slots those are.

The alternative — intersecting the timestamps — was what this notebook used to do, and it
fails in two ways at once. It throws away good frames from the other two series, and it
leaves *holes in the time axis*: NOAA 11536's dopplergram is missing 2012-08-01 09:48 and
2012-08-02 21:48, which put two 1440 s jumps into an otherwise 720 s series. Every FFT in
`src/sunspot_analysis.py` assumes one cadence, so all of them were quietly wrong.

**Caveats this notebook does *not* correct for**, and that you should not read past when
interpreting anything downstream:
- Mean magnetogram values are **signed**. If a box contains both polarities of a bipolar
  group they partially cancel.
- `B_los` changes purely geometrically as the region rotates; no `cos θ` correction is
  applied, so part of any magnetogram trend is projection, not field evolution.
- The umbra mask is *every* pixel below threshold in the box — other spots, pores and bad
  pixels included. There is no connected-component selection of the target spot.
- The continuum stays in **DN/s**: Eq. 3's DN→cgs factor is deliberately not applied,
  since everything measured here is a ratio of intensities.

**Requires cubes built by the current `make_cube`** — older ones have no `TIMESTAMPS`
extension and `read_cube` will refuse them rather than fall back to re-globbing the frame
directory, which cannot detect a missing mid-window frame. The per-frame files in
`data/raw/<region>/region_01/` must also still be there: both the limb-darkening and the
Doppler corrections read the per-frame headers back from them.

In [1]:
import sys
import pathlib

project_root = pathlib.Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root.resolve()))

import numpy as np

from src.utilities import (read_cube, write_cube, regular_time_grid, reindex_on_grid,
                           reindex_series_on_grid, crop_to_common_window)
from src.doppler_calibration import calibrate_cube
from src.limb_darkening import limb_darkening_cube, U_LAMBDA, V_LAMBDA

DATA_DIR      = pathlib.Path('../data/raw')
PROCESSED_DIR = pathlib.Path('../data/processed')

# Segmentation thresholds as fractions of the *per-frame* quiet-sun continuum intensity.
# Absolute DN thresholds don't work here: quiet-sun continuum falls with limb darkening,
# so a fixed cut makes the mask areas drift systematically as the AR rotates, and for a
# near-limb AR the whole frame can fall under a fixed penumbra cut — penumbra becomes the
# entire frame and the quiet-sun mask becomes empty.
UMBRA_FRAC    = 0.60   # I < 0.60 I_qs -> umbra
PENUMBRA_FRAC = 0.90   # I < 0.90 I_qs (and not umbra) -> penumbra
QSUN_PERCENTILE = 80   # percentile of finite pixels used as the frame's I_qs estimate

# --- Hot spot ---------------------------------------------------------------
# The strong-field core inside the sunspot footprint: hot_spot = mag_filter(B) & (umbra|penumbra),
# matching src/sunspot_analysis.py's masks_from_cubes so the downstream metrics and plots
# treat it identically to the sebastian_sun_spots DS regions.
#
# The default is on |B| rather than signed B on purpose. The DS notebooks used `b > 500`
# or `b < -500` chosen per region, and getting the sign wrong yields an *empty* mask
# rather than an error — NOAA 11536's umbra averages about -400 G, so `b > 500` would
# select nothing at all there. Override per AR only when you deliberately want to isolate
# one polarity of a bipolar group.
HOTSPOT_THRESHOLD = 500          # gauss
hotspot_override  = {
    # 11536: lambda b: b < -500,   # negative polarity only
}

# --- Cropping to the common data window -------------------------------------
# A tracked cutout is supposed to keep a constant pixel size for the whole window, so
# normally there is nothing to crop and this stays off. NOAA 11117 is the exception on
# two counts at once:
#
#   - its box shrinks partway through the window, and make_cube center-crops/NaN-pads the
#     smaller frames onto the majority 433x433 grid — that NaN border is never selected by
#     any threshold, so the mask areas step down for exactly as long as the smaller box
#     lasts, which looks like the spot shrinking;
#   - its magnetogram was downloaded under a different box again (402x402), so the three
#     cubes cannot even be indexed against each other.
#
# With the flag on, crop_to_common_window trims all three cubes to the largest rectangle
# in which *every* frame of *every* series has data, so masks built from the continuum
# apply cleanly to the other two and the areas measure the spot rather than the box. The
# saved cubes' CRPIX is shifted to match, so DS9 still puts them on the right sky.
#
# This is a repair for a bad download, not part of the pipeline: the real fix is to
# re-download 11117 with one consistent box.
crop_override = {
    11117: True,
}

# --- Dopplergram zero point -------------------------------------------------
# CALIBRATE_DOPPLER applies the four physical corrections of Castellanos Durán et al.
# (2021) Sect. 2 — observatory velocity, large-scale flows, convective blueshift CLV,
# gravitational redshift — giving velocities on an absolute m/s scale.
#
# RESIDUAL_PLANE_FIT additionally fits and subtracts a plane over the quiet-sun pixels.
# That is the *empirical* correction this notebook used to rely on, and it is off by
# default on purpose: it flattens whatever gradient is left, which also throws away the
# absolute scale the physical corrections just established. Turn it on only to inspect
# what the physical model failed to remove, and don't read absolute velocities off the
# result when you do.
CALIBRATE_DOPPLER  = True
RESIDUAL_PLANE_FIT = False

# --- Continuum limb darkening -----------------------------------------------
# The fifth cleaning step, Castellanos Durán & Kleint (2020) Eqs. 1-2:
#
#     I_corrected = I_observed / C(mu),   C = 1 - u - v + u mu + v mu^2
#
# with u = 0.836, v = -0.204 at the 6173.3 A line HMI observes (src/limb_darkening.py).
# C is bounded in [0.368, 1], so this only ever brightens and can never flip a sign.
#
# **Continuum only.** Limb darkening is an intensity effect; the magnetogram and
# dopplergram are not divided by C — their zero-point corrections are the four above.
#
# It is not made redundant by thresholding each frame against its own quiet-sun
# percentile. That removes the frame-to-frame level drift but nothing inside a frame,
# and across NOAA 11536's box C spans 3-11%, so the umbra/penumbra cut otherwise sits
# systematically tighter on the limbward side by an amount that itself changes as the
# region rotates.
#
# MU_METHOD picks how cos(Theta) is obtained: 'geometry' goes through sunpy's transforms
# (finite observer distance), 'paper' is Eq. 2 literally (observer at infinity, ~3x
# faster, ~0.1% intensity difference near the limb). See src/limb_darkening.mu_from_map.
#
# Eq. 3's DN -> cgs factor (DN_TO_CGS = 52.5) is deliberately NOT applied: every quantity
# downstream is a *ratio* of intensities, so a global scale cancels out of all of them
# while making every number harder to compare against the raw frames and against DS9.
CORRECT_LIMB_DARKENING = True
MU_METHOD              = 'geometry'

# The magnetogram is unaffected by any of the above — those are Doppler zero-point and
# intensity effects — but still carries an instrumental offset, so it keeps the quiet-sun
# plane subtraction unconditionally.

# Mask bit assignments, also written into the FITS header of the mask cube.
BIT_UMBRA, BIT_PENUMBRA, BIT_HOTSPOT = 1, 2, 4

## Discover regions

Any `NOAA_*` directory with a `region_01_continuum_cube.fits` — this naturally skips the pre-HMI ARs (11039, 11041), which have no cube files at all.

In [2]:
regions = []
for region_dir in sorted(DATA_DIR.glob('NOAA_*')):
    cont_cube = region_dir / 'region_01_continuum_cube.fits'
    if cont_cube.exists():
        regions.append(region_dir)

print(f'{len(regions)} region(s) found:')
for r in regions:
    print(' ', r.name)

4 region(s) found:
  NOAA_11106_2010-09-16
  NOAA_11117_2010-10-27
  NOAA_11363_2011-12-06
  NOAA_11536_2012-07-31


## Per-region processing

`process_region` loads the three cubes, corrects them, and places them on **one uniform
time grid** built from the union of their timestamps. Individual JSOC files do fail, and
when a frame is missing mid-window — which the download logs show happens — the slot stays
in the axis and its frame is NaN, rather than the frame being dropped and every later frame
sliding one slot out of step.

It then:

- divides the continuum by `C(mu)` through `limb_darkening_cube`, which reads the per-frame
  headers back from the frame directory (the box is tracked, so `mu` changes every frame);
- calibrates the dopplergram through `calibrate_cube`, which reads those same headers back
  for the same reason (`OBS_V*` changes every frame);
- segments each continuum frame against that frame's *own* quiet-sun intensity
  (`UMBRA_FRAC`/`PENUMBRA_FRAC` × `I_qs`) instead of an absolute DN cut;
- subtracts a fitted **plane** from the magnetogram rather than a scalar mean, which takes
  out the gradient across the box and not just the uniform offset;
- builds the **hot spot** inside the sunspot footprint.

Both corrections run *before* the grid join, on the frames that exist; their per-frame
diagnostics (`c_means`, `doppler_terms`) are moved onto the grid alongside the cubes.

A gap slot flows through every step as an all-NaN frame and comes out with empty masks, so
`present` is carried through to tell those apart from a frame in which the spot genuinely
wasn't detected.

It returns the corrected cubes and masks. It computes no time series and draws nothing —
`03A_data_analisis.ipynb` derives all of that from the written cubes, so the two notebooks
can't drift apart.

In [3]:
def load_aligned(region_dir, calibrate=CALIBRATE_DOPPLER, limb_darken=CORRECT_LIMB_DARKENING,
                 crop_to_data=False, v_sdo_by_time=None):
    """Load the three cubes, correct them, and place them on one uniform time grid.

    The three series are downloaded independently and individual files do fail, so their
    frame counts differ. Two joins were considered and only one of them is safe:

    - *Intersection* of the timestamps (what this notebook used to do). It keeps only
      frames all three series have, which throws away good frames from the others and,
      worse, leaves holes in the time axis. NOAA 11536's dopplergram is missing
      2012-08-01 09:48 and 2012-08-02 21:48, so the intersection has two 1440 s jumps in
      an otherwise 720 s series — and every FFT in src/sunspot_analysis.py assumes a
      single cadence, so all of them come out quietly wrong.
    - A *uniform grid* covering the union of the timestamps, which is what happens here.
      Every series gets one frame per grid slot; where a series has no frame the slot is
      NaN. The time axis is then uniform by construction rather than inherited from
      whatever happened to download, and a missing mid-window frame stays visible as a
      gap instead of shifting every later frame out of step.

    Both corrections are applied *before* the join and per frame, because both depend on
    where the tracked box is pointing: the OBS_V* keywords change every frame, and mu at
    the box centre runs 0.78 -> 0.87 -> 0.85 across NOAA 11536's window. calibrate_cube
    and limb_darkening_cube therefore read the per-frame headers back from the region's
    frame directory (make_cube keeps only the reference frame's header).

    Parameters
    ----------
    crop_to_data : bool
        Trim all three cubes to the window in which every frame has data — see
        `crop_to_common_window` and the `crop_override` dict below. Off by default because
        it is a repair for a bad download, not part of the pipeline: on a healthy region it
        finds nothing to trim and only costs a pass over the cubes.

    Returns
    -------
    dict with keys
        cubes         {'cont', 'mag', 'dop'}, each (n_slots, ny, nx) float32
        grid          list of datetimes, one per slot, evenly spaced
        cadence_s     the grid spacing
        present       {'cont', 'mag', 'dop'} bool arrays — True where the series has data
        doppler_terms per-term spatial means on the grid, or None
        c_means       per-frame spatial mean of the limb-darkening factor C, or None
        crop_offset   (row0, col0) of the crop in the continuum's original pixels, or None
    """
    cubes, times = {}, {}
    for name, fname in [('cont', 'continuum'), ('mag', 'magnetogram'), ('dop', 'dopplergram')]:
        data, ts = read_cube(region_dir / f'region_01_{fname}_cube.fits')
        cubes[name], times[name] = data.astype(np.float32), ts

    c_means = None
    if limb_darken:
        corrected, cont_times, c_means = limb_darkening_cube(
            region_dir, method=MU_METHOD, u_lambda=U_LAMBDA, v_lambda=V_LAMBDA)
        if cont_times != times['cont']:
            raise ValueError('limb_darkening_cube returned different timestamps than the cube')
        cubes['cont'] = corrected

    term_means = None
    if calibrate:
        corrected, dop_times, term_means = calibrate_cube(region_dir, v_sdo_by_time=v_sdo_by_time)
        if dop_times != times['dop']:
            raise ValueError('calibrate_cube returned different timestamps than the cube')
        cubes['dop'] = corrected

    grid, cadence_s = regular_time_grid([times['cont'], times['mag'], times['dop']])

    present = {}
    for name in cubes:
        cubes[name], present[name] = reindex_on_grid(cubes[name], times[name], grid, cadence_s)

    # The per-frame diagnostics have to move onto the same grid as the cubes they describe,
    # or they end up plotted against the wrong times.
    if term_means is not None:
        term_means = {k: reindex_series_on_grid(v, times['dop'], grid, cadence_s)
                      for k, v in term_means.items()}
    if c_means is not None:
        c_means = reindex_series_on_grid(c_means, times['cont'], grid, cadence_s)

    gaps = {k: np.flatnonzero(~v) for k, v in present.items()}
    n_gaps = {k: len(v) for k, v in gaps.items()}
    print(f'  {len(grid)} slots on a uniform {cadence_s:.0f} s grid, '
          f'{grid[0]:%Y-%m-%d %H:%M} .. {grid[-1]:%Y-%m-%d %H:%M}')
    if any(n_gaps.values()):
        print(f'  NaN frames (no data): {n_gaps}')
        for name, idx in gaps.items():
            for i in idx[:5]:
                print(f'      {name}: slot {i} = {grid[i]:%Y-%m-%d %H:%M}')
            if len(idx) > 5:
                print(f'      {name}: ... and {len(idx) - 5} more')
    else:
        print('  no gaps — all three series cover every slot')

    crop_offset = None
    if crop_to_data:
        cubes, offsets = crop_to_common_window(cubes, present=present)
        crop_offset = offsets['cont']
    else:
        # Everything downstream indexes the magnetogram and dopplergram with masks built
        # from the continuum, so a shape mismatch has to stop here with an explanation
        # rather than as a broadcasting error 60 lines later.
        shapes = {name: cube.shape[1:] for name, cube in cubes.items()}
        if len(set(shapes.values())) > 1:
            raise ValueError(
                f'{region_dir.name}: the three series are on different pixel grids '
                f'({shapes}) — they were downloaded under different boxes. Either '
                f're-download the region with one consistent box, or set this region to '
                f'True in crop_override to trim all three to their common data window.')

    return dict(cubes=cubes, grid=grid, cadence_s=cadence_s, present=present,
                doppler_terms=term_means, c_means=c_means, crop_offset=crop_offset)


def remove_quiet_sun_plane(frame, qsun_mask, xn, yn):
    """Fit a plane to the quiet-sun pixels and subtract it from the whole frame.

    Subtracting a scalar quiet-sun mean only removes the spatially uniform term — for the
    dopplergram that is mostly the SDO orbital velocity. What survives is the line-of-sight
    solar-rotation gradient across the box, and because the umbra sits off to one side of
    the box its mean picks up a residual that drifts as the region rotates. Removing a
    plane instead takes out that gradient to first order.

    This is the *empirical* alternative to the physical corrections in
    src/doppler_calibration.py. It always applies to the magnetogram (which those
    corrections don't address) but only to the dopplergram when RESIDUAL_PLANE_FIT is set.

    A NaN (gap) frame has no finite quiet-sun pixels, so it returns unchanged with NaN
    coefficients rather than raising.

    Returns (corrected_frame, coefficients) with coefficients = (offset, d/dx, d/dy) in
    the normalized coordinates xn, yn.
    """
    valid = qsun_mask & np.isfinite(frame)
    if valid.sum() < 3:
        return frame, np.array([np.nan, np.nan, np.nan])
    design = np.column_stack([np.ones(valid.sum()), xn[valid], yn[valid]])
    coef, *_ = np.linalg.lstsq(design, frame[valid].astype(np.float64), rcond=None)
    plane = coef[0] + coef[1] * xn + coef[2] * yn
    return (frame - plane).astype(np.float32), coef


def process_region(region_dir, noaa=None, crop_to_data=False, v_sdo_by_time=None):
    """Correct one region's cubes and build its masks.

    Returns everything the writer and the summary need; it deliberately computes no plots
    and no time series — 03A_data_analisis.ipynb derives those from the written cubes.

    Every step here is NaN-safe, because a gap slot is an all-NaN frame: the I_qs
    percentile, the plane fit and the quiet-sun means all guard on there being finite
    pixels, and a comparison against a NaN threshold is False, so a gap frame simply gets
    empty masks. `present` is carried through so downstream can tell those apart from a
    frame where the spot genuinely wasn't detected.
    """
    aligned    = load_aligned(region_dir, crop_to_data=crop_to_data, v_sdo_by_time=v_sdo_by_time)
    cubes      = aligned['cubes']
    timestamps = aligned['grid']
    cube_cont, cube_mag, cube_dop = cubes['cont'], cubes['mag'], cubes['dop']
    n_t, ny, nx = cube_cont.shape

    # Normalized pixel coordinates for the plane fit, so the design matrix stays well
    # conditioned regardless of box size.
    yy, xx = np.mgrid[0:ny, 0:nx]
    xn = ((xx - nx / 2) / (nx / 2)).astype(np.float64)
    yn = ((yy - ny / 2) / (ny / 2)).astype(np.float64)

    finite = np.isfinite(cube_cont)
    i_qs = np.array([np.percentile(cube_cont[t][finite[t]], QSUN_PERCENTILE)
                     if finite[t].any() else np.nan for t in range(n_t)])

    # A NaN threshold compares False everywhere, so a gap frame gets empty masks.
    with np.errstate(invalid='ignore'):
        umbra    = (cube_cont < (UMBRA_FRAC * i_qs)[:, None, None]) & finite
        penumbra = (cube_cont < (PENUMBRA_FRAC * i_qs)[:, None, None]) & finite & ~umbra
    both     = umbra | penumbra
    qsun     = finite & ~both

    plane_coefs = {'mag': np.full((n_t, 3), np.nan), 'dop': np.full((n_t, 3), np.nan)}
    cube_mag = cube_mag.copy()          # crop_to_common_window returns views
    cube_dop = cube_dop.copy() if RESIDUAL_PLANE_FIT else cube_dop
    for t in range(n_t):
        cube_mag[t], plane_coefs['mag'][t] = remove_quiet_sun_plane(cube_mag[t], qsun[t], xn, yn)
        if RESIDUAL_PLANE_FIT:
            cube_dop[t], plane_coefs['dop'][t] = remove_quiet_sun_plane(cube_dop[t], qsun[t], xn, yn)

    # Hot spot on the *plane-corrected* magnetogram, so the threshold means the same thing
    # in every frame rather than drifting with the instrumental offset.
    mag_filter = hotspot_override.get(noaa, lambda b: np.abs(b) > HOTSPOT_THRESHOLD)
    with np.errstate(invalid='ignore'):
        hot_spot = mag_filter(cube_mag) & both

    qsun_means = {
        'mag': np.array([np.nanmean(cube_mag[t][qsun[t]]) if qsun[t].any() else np.nan
                         for t in range(n_t)]),
        'dop': np.array([np.nanmean(cube_dop[t][qsun[t]]) if qsun[t].any() else np.nan
                         for t in range(n_t)]),
    }

    masks = {'umbra': umbra, 'penumbra': penumbra, 'hot_spot': hot_spot, 'qsun': qsun}
    return dict(
        cubes={'cont': cube_cont, 'mag': cube_mag, 'dop': cube_dop},
        masks=masks, timestamps=timestamps, i_qs=i_qs, qsun_means=qsun_means,
        plane_coefs=plane_coefs, doppler_terms=aligned['doppler_terms'],
        c_means=aligned['c_means'], present=aligned['present'],
        cadence_s=aligned['cadence_s'], crop_offset=aligned['crop_offset'],
        area_px={name: m.reshape(n_t, -1).sum(axis=1) for name, m in masks.items()},
    )

In [4]:
from astropy.io import fits


def region_noaa(region_dir):
    """NOAA number from a `NOAA_<number>_<date>` directory name, or None."""
    parts = region_dir.name.split('_')
    return int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else None


def write_region(region_dir, result):
    """Write one region's corrected cubes, masks and quiet-sun means.

    Everything goes out through src.utilities.write_cube so it comes back through
    read_cube unchanged, and so DS9 sees the same structure as the raw cubes: a 3D
    primary HDU carrying the reference frame's spatial WCS, plus a TIMESTAMPS extension.
    """
    out_dir = PROCESSED_DIR / region_dir.name
    timestamps = result['timestamps']
    present    = result['present']
    cadence_s  = result['cadence_s']
    n_gaps     = int((~(present['cont'] & present['mag'] & present['dop'])).sum())

    # Reuse the raw continuum cube's header so the corrected cubes keep the spatial WCS,
    # and DS9 puts them on the same footprint as the frames they came from.
    src_header = fits.getheader(region_dir / 'region_01_continuum_cube.fits')
    for key in ('NAXIS', 'NAXIS1', 'NAXIS2', 'NAXIS3', 'NFRAMES', 'BITPIX'):
        src_header.pop(key, None)
    src_header['CADENCE'] = (cadence_s, '[s] uniform grid spacing along axis 3')
    src_header['NGAPS']   = (n_gaps, 'frames with no data in at least one series')

    # A crop moves the origin, so the inherited CRPIX would put the cube in the wrong place
    # on the sky. FITS pixel coordinates are 1-based and the crop offset is 0-based, but
    # both refer to the same axis, so the shift is just a subtraction.
    crop_offset = result.get('crop_offset')
    if crop_offset is not None:
        row0, col0 = crop_offset
        src_header['CRPIX1'] = src_header['CRPIX1'] - col0
        src_header['CRPIX2'] = src_header['CRPIX2'] - row0
        src_header['CROPROW'] = (row0, 'first row of the crop in the original cube')
        src_header['CROPCOL'] = (col0, 'first column of the crop in the original cube')

    custom_hotspot = region_noaa(region_dir) in hotspot_override
    provenance = [
        f'02A: {len(timestamps)} frames on a uniform {cadence_s:.0f} s grid, '
        f'{n_gaps} NaN (missing) frame(s)',
        f'02A: umbra < {UMBRA_FRAC} I_qs, penumbra < {PENUMBRA_FRAC} I_qs (I_qs = p{QSUN_PERCENTILE})',
        '02A: hot spot from a per-AR hotspot_override filter' if custom_hotspot else
        f'02A: hot spot |B| > {HOTSPOT_THRESHOLD} G within the sunspot',
    ]
    if crop_offset is not None:
        ny, nx = result['cubes']['cont'].shape[1:]
        provenance.append(f'02A: cropped to the common data window {ny}x{nx} at '
                          f'row {crop_offset[0]}, col {crop_offset[1]}; CRPIX shifted to match')

    doppler_history = (
        ['02A: dopplergram calibrated - Castellanos Duran 2021 sdo+lsf+clv+gravity']
        if CALIBRATE_DOPPLER else ['02A: dopplergram NOT calibrated'])
    if RESIDUAL_PLANE_FIT:
        doppler_history.append('02A: residual quiet-sun plane also subtracted (absolute scale lost)')

    if CORRECT_LIMB_DARKENING:
        continuum_history = [
            '02A: continuum limb-darkening corrected - Castellanos Duran & Kleint 2020 '
            f'Eq.1-2 (u={U_LAMBDA}, v={V_LAMBDA}, mu from {MU_METHOD})',
            '02A: intensity left in DN/s - Eq.3 DN->cgs factor NOT applied']
    else:
        continuum_history = ['02A: continuum NOT limb-darkening corrected']

    cont_header = src_header.copy()
    cont_header['LDCORR'] = (CORRECT_LIMB_DARKENING, 'limb darkening divided out (Duran 2020 Eq.1)')
    if CORRECT_LIMB_DARKENING:
        cont_header['LD_U']   = (U_LAMBDA, 'limb-darkening coefficient u at 6173.3 A')
        cont_header['LD_V']   = (V_LAMBDA, 'limb-darkening coefficient v at 6173.3 A')
        cont_header['LD_MU']  = (MU_METHOD, 'how cos(Theta) was obtained')

    written = {}
    written['continuum'] = write_cube(
        result['cubes']['cont'], out_dir / 'region_01_continuum_cube.fits',
        header=cont_header, timestamps=timestamps,
        history=provenance + continuum_history)

    written['magnetogram'] = write_cube(
        result['cubes']['mag'], out_dir / 'region_01_magnetogram_corrected_cube.fits',
        header=src_header, timestamps=timestamps,
        history=provenance + ['02A: quiet-sun plane subtracted from the magnetogram'])

    written['dopplergram'] = write_cube(
        result['cubes']['dop'], out_dir / 'region_01_dopplergram_calibrated_cube.fits',
        header=src_header, timestamps=timestamps, history=provenance + doppler_history)

    # Masks as one uint8 bitmask: DS9 renders integers far more cleanly than floats, and a
    # single file keeps the overlapping hot spot alongside the regions it sits inside.
    # The bit values and threshold fractions are keywords rather than a convention, so
    # load_noaa_region reads back exactly what was used instead of assuming defaults.
    m = result['masks']
    bitmask = (m['umbra'].astype(np.uint8) * BIT_UMBRA
               | m['penumbra'].astype(np.uint8) * BIT_PENUMBRA
               | m['hot_spot'].astype(np.uint8) * BIT_HOTSPOT)
    mask_header = src_header.copy()
    mask_header['BUNIT']    = ('', 'bit flags, see BIT_* keywords')
    mask_header['BIT_UMB']  = (BIT_UMBRA, 'bit value for umbra')
    mask_header['BIT_PEN']  = (BIT_PENUMBRA, 'bit value for penumbra')
    mask_header['BIT_HOT']  = (BIT_HOTSPOT, 'bit value for hot spot')
    mask_header['UMB_FRAC'] = (UMBRA_FRAC, 'umbra threshold as a fraction of I_qs')
    mask_header['PEN_FRAC'] = (PENUMBRA_FRAC, 'penumbra threshold as a fraction of I_qs')
    written['masks'] = write_cube(
        bitmask, out_dir / 'region_01_masks_cube.fits',
        header=mask_header, timestamps=timestamps,
        history=provenance + ['02A: 0 = quiet sun; bits overlap (hot spot is inside the spot)',
                              '02A: a gap frame has every bit 0 - see PRESENT_* in the qsun table'])

    # Quiet-sun means as a small table. compute_metrics only ever takes their per-frame
    # mean, so saving the numbers avoids carrying two more full-size cubes around.
    #
    # The PRESENT_* flags live here because there is nowhere else for them: the mask cube
    # is uint8 and cannot hold a NaN, so without them a gap frame's empty masks look
    # exactly like a frame in which the spot was not detected. C_MEAN rides along for the
    # same reason the doppler terms are reported — a correction whose size cannot be
    # inspected is indistinguishable from solar signal when it goes wrong.
    n_t = len(timestamps)
    c_means = result['c_means']
    if c_means is None:
        c_means = np.full(n_t, np.nan)
    cols = [
        fits.Column(name='T_OBS', format='23A',
                    array=np.array([t.isoformat() for t in timestamps])),
        fits.Column(name='MEAN_MAG_QSUN', format='E', array=result['qsun_means']['mag']),
        fits.Column(name='MEAN_DOP_QSUN', format='E', array=result['qsun_means']['dop']),
        fits.Column(name='I_QS', format='E', array=result['i_qs']),
        fits.Column(name='C_MEAN', format='E', array=c_means),
        fits.Column(name='PRESENT_CONT', format='L', array=present['cont']),
        fits.Column(name='PRESENT_MAG', format='L', array=present['mag']),
        fits.Column(name='PRESENT_DOP', format='L', array=present['dop']),
    ]
    qsun_path = out_dir / 'region_01_qsun_means.fits'
    qsun_path.parent.mkdir(parents=True, exist_ok=True)
    fits.BinTableHDU.from_columns(cols, name='QSUN').writeto(qsun_path, overwrite=True)
    written['qsun_means'] = str(qsun_path)

    return out_dir, written

In [ ]:
results = {}
for region_dir in regions:
    print(region_dir.name)
    noaa = region_noaa(region_dir)
    result = process_region(region_dir, noaa=noaa,
                            crop_to_data=crop_override.get(noaa, False))
    out_dir, written = write_region(region_dir, result)
    results[region_dir.name] = result

    n_t = len(result['timestamps'])
    area = result['area_px']
    present = result['present']
    span_h = (result['timestamps'][-1] - result['timestamps'][0]).total_seconds() / 3600
    print(f'  {n_t} frames over {span_h:.1f} h, box {result["cubes"]["cont"].shape[1:]} px')
    # Areas are counted over the frames that actually have data — a gap frame has empty
    # masks by construction and would otherwise drag every mean down.
    with_data = present['cont']
    print(f'  mean area px  umbra={area["umbra"][with_data].mean():.0f}  '
          f'penumbra={area["penumbra"][with_data].mean():.0f}  '
          f'hot spot={area["hot_spot"][with_data].mean():.0f}  '
          f'quiet sun={area["qsun"][with_data].mean():.0f}')
    # An empty hot spot is the failure mode to watch for: a polarity-specific override
    # silently selects nothing when the spot has the other sign. Gap frames are excluded,
    # since those are empty for a reason that has nothing to do with the filter.
    empty = int(((area['hot_spot'] == 0) & with_data).sum())
    if empty:
        print(f'  WARNING: hot spot is empty in {empty}/{int(with_data.sum())} frame(s) with data '
              f'— check the filter for NOAA {noaa} against the actual polarity')
    print(f'  quiet sun  <B>={np.nanmean(result["qsun_means"]["mag"]):7.1f} G   '
          f'<v>={np.nanmean(result["qsun_means"]["dop"]):7.1f} m/s')
    if result['c_means'] is not None:
        c = result['c_means']
        print(f'  limb darkening  <C>={np.nanmean(c):.4f}  '
              f'({np.nanmin(c):.4f} .. {np.nanmax(c):.4f}, so I is boosted by up to '
              f'{1 / np.nanmin(c):.3f}x)')
    print(f'  -> {out_dir}')
    for name, path in written.items():
        print(f'       {name:12s} {pathlib.Path(path).name}')

NOAA_11117_2010-10-27
  960 slots on a uniform 360 s grid, 2010-10-27 00:00 .. 2010-10-30 23:54
  NaN frames (no data): {'cont': 131, 'mag': 456, 'dop': 131}
      cont: slot 390 = 2010-10-28 15:00
      cont: slot 391 = 2010-10-28 15:06
      cont: slot 393 = 2010-10-28 15:18
      cont: slot 397 = 2010-10-28 15:42
      cont: slot 398 = 2010-10-28 15:48
      cont: ... and 126 more
      mag: slot 64 = 2010-10-27 06:24
      mag: slot 66 = 2010-10-27 06:36
      mag: slot 68 = 2010-10-27 06:48
      mag: slot 70 = 2010-10-27 07:00
      mag: slot 72 = 2010-10-27 07:12
      mag: ... and 451 more
      dop: slot 390 = 2010-10-28 15:00
      dop: slot 391 = 2010-10-28 15:06
      dop: slot 393 = 2010-10-28 15:18
      dop: slot 397 = 2010-10-28 15:42
      dop: slot 398 = 2010-10-28 15:48
      dop: ... and 126 more
  cropped to the common data window: cont=433x433, mag=402x402, dop=433x433 -> 402x402 (100% of the smallest input box)


/tmp/ipykernel_29680/1112299529.py:191: RuntimeWarning: Mean of empty slice
  'mag': np.array([np.nanmean(cube_mag[t][qsun[t]]) if qsun[t].any() else np.nan


  960 frames over 95.9 h, box (402, 402) px
  mean area px  umbra=2369  penumbra=11705  hot spot=3128  quiet sun=146830
  quiet sun  <B>=   -0.0 G   <v>= -313.8 m/s
  limb darkening  <C>=0.8621  (0.6570 .. 0.9662, so I is boosted by up to 1.522x)
  -> ../data/processed/NOAA_11117_2010-10-27
       continuum    region_01_continuum_cube.fits
       magnetogram  region_01_magnetogram_corrected_cube.fits
       dopplergram  region_01_dopplergram_calibrated_cube.fits
       masks        region_01_masks_cube.fits
       qsun_means   region_01_qsun_means.fits


TODO: make simplier the cropout part or just redowload the things i need

## Diagnostics — was the signal solar or instrumental?

A numeric summary of what was removed and what the segmentation did. The corresponding
plots live in `03A_data_analisis.ipynb`; what matters here is that nothing looks absurd
before the cubes get written.

- **`I_qs`** should drift smoothly and slowly. Most of its old drift was limb darkening and
  is now divided out, so what is left is the correction's residual plus real evolution. The
  point of *also* thresholding against it is that the mask areas should not inherit any of
  that, so a large `I_qs` swing next to a flat area is the correct outcome.
- **`C`** should sweep by several percent across the window as the region rotates, and
  should be smallest (strongest correction) when the region is nearest the limb. A flat `C`
  means the per-frame headers aren't being read and every frame got the same correction.
- **Mask areas** should be roughly flat. A monotonic ramp means the segmentation is still
  tracking something geometric rather than the spot; a step means frames of mixed
  provenance. Gap frames are excluded from these statistics — their masks are empty by
  construction, not by measurement.
- **Doppler terms** are what the old empirical plane fit used to absorb blindly. `sdo`
  swings by hundreds of m/s over a day; if it doesn't, the per-frame headers aren't being
  read.

In [7]:
def summarize(region_name, result):
    n_t = len(result['timestamps'])
    present = result['present']
    with_data = present['cont']
    n_gaps = {k: int((~v).sum()) for k, v in present.items()}
    print(f'\n{region_name}  ({n_t} slots @ {result["cadence_s"]:.0f} s, gaps {n_gaps})')

    i_qs = result['i_qs']
    print(f'  I_qs           {np.nanmin(i_qs):8.0f} .. {np.nanmax(i_qs):8.0f} DN   '
          f'({100 * np.ptp(i_qs[np.isfinite(i_qs)]) / np.nanmean(i_qs):.1f}% drift)')

    c = result['c_means']
    if c is not None:
        finite_c = c[np.isfinite(c)]
        print(f'  limb darkening C  {finite_c.min():.4f} .. {finite_c.max():.4f}   '
              f'(mean {finite_c.mean():.4f}; a flat C means the per-frame headers '
              f'are not being read)')

    # Drift over the frames that have data. A gap frame contributes area 0 and would
    # otherwise show up as a 100% drift in every region.
    print('  mask area (px)      mean     min     max   drift')
    for name, area in result['area_px'].items():
        a = area[with_data]
        drift = 100 * np.ptp(a) / a.mean() if a.mean() else np.nan
        print(f'    {name:12s} {a.mean():8.0f} {a.min():7.0f} {a.max():7.0f} {drift:6.1f}%')

    terms = result['doppler_terms']
    if terms:
        print('  Doppler terms removed (m/s)     mean   peak-to-peak')
        for name, series in terms.items():
            s = series[np.isfinite(series)]
            print(f'    {name:10s} {s.mean():20.1f} {np.ptp(s):10.1f}')

    coefs = result['plane_coefs']['mag']
    if np.isfinite(coefs).any():
        grad = np.hypot(coefs[:, 1], coefs[:, 2])
        print(f'  magnetogram plane |gradient|  {np.nanmean(grad):.1f} G per half-box')


for region_name, result in results.items():
    summarize(region_name, result)


NOAA_11117_2010-10-27  (960 slots @ 360 s, gaps {'cont': 131, 'mag': 456, 'dop': 131})
  I_qs              69694 ..    70542 DN   (1.2% drift)
  limb darkening C  0.6570 .. 0.9662   (mean 0.8621; a flat C means the per-frame headers are not being read)
  mask area (px)      mean     min     max   drift
    umbra            2369     826    3154   98.3%
    penumbra        11705    5995   15014   77.0%
    hot_spot         3128       0    7990  255.5%
    qsun           146830  133618  154136   14.0%
  Doppler terms removed (m/s)     mean   peak-to-peak
    sdo                      -503.0     5655.8
    lsf                      1116.0     1204.5
    clv                      -351.8       98.9
    gravity                   636.0        0.0
  magnetogram plane |gradient|  31.2 G per half-box
